# Wan2.1-VACE vs HunyuanVideo: Human Video Editing Comparison
**Runtime**: A100 40GB recommended (Colab Pro)
**Time**: ~10-15 minutes total

In [ ]:
# Check GPU
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Install dependencies
!pip install -q "diffusers>=0.33.0" transformers accelerate peft
!pip install -q imageio imageio-ffmpeg safetensors sentencepiece protobuf ftfy

In [ ]:
import torch, time, json, os
from pathlib import Path

PROMPTS = [
    "A young woman with dark hair turns to face the camera and smiles, cinematic lighting, 4K",
    "A man in a business suit walks confidently down a city street, shallow depth of field",
    "A person with curly red hair dances in a sunlit room, natural lighting, portrait style",
]

os.makedirs("results/wan/generation", exist_ok=True)
os.makedirs("results/hunyuan/generation", exist_ok=True)
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

## 1. Wan2.1-T2V-1.3B — Human Video Generation

In [ ]:
from diffusers import WanPipeline
from diffusers.utils import export_to_video

pipe_wan = WanPipeline.from_pretrained(
    "Wan-AI/Wan2.1-T2V-1.3B-Diffusers",
    torch_dtype=torch.float16,
)
pipe_wan.to("cuda")
print(f"Wan loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
wan_results = []
for i, prompt in enumerate(PROMPTS):
    print(f"\n[{i+1}/3] {prompt[:60]}...")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    out = pipe_wan(
        prompt=prompt, num_frames=17, height=480, width=832,
        num_inference_steps=30, guidance_scale=5.0,
        generator=torch.Generator("cuda").manual_seed(42),
    )
    dt = time.time() - t0
    peak = torch.cuda.max_memory_allocated() / 1e9
    path = f"results/wan/generation/gen_{i:02d}.mp4"
    export_to_video(out.frames[0], path, fps=8)
    wan_results.append({"time": round(dt,1), "vram": round(peak,2), "path": path})
    print(f"  Done: {dt:.1f}s, VRAM: {peak:.2f} GB")

del pipe_wan; torch.cuda.empty_cache()
print("\nWan results:", wan_results)

## 2. Wan2.1-VACE-1.3B — Video Editing

In [ ]:
from diffusers import WanVACEPipeline

pipe_vace = WanVACEPipeline.from_pretrained(
    "Wan-AI/Wan2.1-VACE-1.3B-diffusers",
    torch_dtype=torch.float16,
)
pipe_vace.to("cuda")
print(f"VACE loaded. VRAM: {torch.cuda.memory_allocated()/1e9:.2f} GB")

In [ ]:
EDITS = [
    {"name": "hair_blonde", "prompt": "A person with bright blonde hair, same pose, same background"},
    {"name": "sunglasses", "prompt": "A person wearing stylish dark sunglasses, same pose"},
    {"name": "smile", "prompt": "A person with a big warm smile, happy expression"},
    {"name": "painting", "prompt": "An oil painting of a person, impressionist, brush strokes"},
    {"name": "beach_bg", "prompt": "A person standing on a tropical beach with ocean waves"},
]

os.makedirs("results/wan/editing", exist_ok=True)
vace_results = []
for edit in EDITS:
    print(f"\nEdit: {edit['name']}...")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        out = pipe_vace(
            prompt=edit["prompt"], num_frames=17, height=480, width=832,
            num_inference_steps=30, guidance_scale=5.0,
            generator=torch.Generator("cuda").manual_seed(42),
        )
        dt = time.time() - t0
        peak = torch.cuda.max_memory_allocated() / 1e9
        path = f"results/wan/editing/edit_{edit['name']}.mp4"
        export_to_video(out.frames[0], path, fps=8)
        vace_results.append({"name": edit["name"], "status": "ok", "time": round(dt,1), "vram": round(peak,2)})
        print(f"  Done: {dt:.1f}s, VRAM: {peak:.2f} GB")
    except Exception as e:
        vace_results.append({"name": edit["name"], "status": "error", "error": str(e)})
        print(f"  Failed: {e}")

del pipe_vace; torch.cuda.empty_cache()
print("\nVACE results:", vace_results)

## 3. HunyuanVideo — Human Video Generation

In [ ]:
from diffusers import HunyuanVideoPipeline

pipe_hv = HunyuanVideoPipeline.from_pretrained(
    "hunyuanvideo-community/HunyuanVideo",
    torch_dtype=torch.float16,
)
# Apply offloading FIRST
pipe_hv.enable_model_cpu_offload()
# THEN force VAE to fp32 (offload resets dtypes, so this must come after)
pipe_hv.vae = pipe_hv.vae.to(dtype=torch.float32)
if hasattr(pipe_hv.vae, 'config'):
    pipe_hv.vae.config.force_upcast = True
pipe_hv.vae.enable_tiling()
pipe_hv.vae.enable_slicing()
print("HunyuanVideo loaded with CPU offload + fp32 VAE")

In [ ]:
hv_results = []
for i, prompt in enumerate(PROMPTS):
    print(f"\n[{i+1}/3] {prompt[:60]}...")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    try:
        out = pipe_hv(
            prompt=prompt, num_frames=17, height=480, width=832,
            num_inference_steps=30, guidance_scale=6.0,
            generator=torch.Generator("cpu").manual_seed(42),
        )
        dt = time.time() - t0
        peak = torch.cuda.max_memory_allocated() / 1e9
        path = f"results/hunyuan/generation/gen_{i:02d}.mp4"
        export_to_video(out.frames[0], path, fps=8)
        fsize = os.path.getsize(path)
        hv_results.append({"time": round(dt,1), "vram": round(peak,2), "path": path, "size_kb": round(fsize/1024,1)})
        print(f"  Done: {dt:.1f}s, VRAM: {peak:.2f} GB, file: {fsize/1024:.1f} KB")
    except Exception as e:
        hv_results.append({"status": "error", "error": str(e)})
        print(f"  Failed: {e}")
        torch.cuda.empty_cache()

del pipe_hv; torch.cuda.empty_cache()
print("\nHunyuanVideo results:", hv_results)

## 4. LoRA Compatibility Comparison

In [ ]:
import torch.nn as nn

def probe_lora(model_id, pipeline_cls):
    pipe = pipeline_cls.from_pretrained(model_id, torch_dtype=torch.float16)
    t = pipe.transformer
    linears = [(n, m.in_features, m.out_features) for n, m in t.named_modules() if isinstance(m, nn.Linear)]
    attns = [n for n, _, _ in linears if "attn" in n.lower()]
    total = sum(m.weight.numel() for _, m in t.named_modules() if isinstance(m, nn.Linear))
    sizes = {}
    for r in [1, 4, 8, 16]:
        p = sum((inf * r + r * outf) for _, inf, outf in linears)
        sizes[f"rank_{r}"] = f"{p * 2 / 1024 / 1024:.1f} MB"
    del pipe; torch.cuda.empty_cache()
    return {"params_m": round(total/1e6,1), "linear": len(linears), "attn": len(attns),
            "has_lora": True, "lora_sizes": sizes}

print("Probing Wan2.1...")
wan_lora = probe_lora("Wan-AI/Wan2.1-T2V-1.3B-Diffusers", WanPipeline)
print(f"  Wan: {wan_lora}")

print("\nProbing HunyuanVideo...")
hv_lora = probe_lora("hunyuanvideo-community/HunyuanVideo", HunyuanVideoPipeline)
print(f"  HunyuanVideo: {hv_lora}")

## 5. Summary & Side-by-Side

In [ ]:
from IPython.display import HTML, display
import base64

def video_tag(path, label):
    if not os.path.exists(path) or os.path.getsize(path) < 1000:
        return f"<div><b>{label}</b><br>Video failed/empty</div>"
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'''<div style="display:inline-block;margin:10px;text-align:center">
        <b>{label}</b><br>
        <video width="400" controls autoplay loop muted>
            <source src="data:video/mp4;base64,{b64}" type="video/mp4">
        </video></div>'''

for i, prompt in enumerate(PROMPTS):
    display(HTML(f"<h3>Prompt {i+1}: {prompt[:60]}...</h3>"))
    html = video_tag(f"results/wan/generation/gen_{i:02d}.mp4", "Wan2.1-1.3B")
    html += video_tag(f"results/hunyuan/generation/gen_{i:02d}.mp4", "HunyuanVideo")
    display(HTML(html))

print("\n=== COMPARISON TABLE ===")
print(f"{'':20s} {'Wan2.1-1.3B':>15s} {'HunyuanVideo':>15s}")
print(f"{'Params':20s} {wan_lora['params_m']:>14.1f}M {hv_lora['params_m']:>14.1f}M")
print(f"{'Linear layers':20s} {wan_lora['linear']:>15d} {hv_lora['linear']:>15d}")
print(f"{'LoRA rank-4':20s} {wan_lora['lora_sizes']['rank_4']:>15s} {hv_lora['lora_sizes']['rank_4']:>15s}")
print(f"{'LoRA rank-16':20s} {wan_lora['lora_sizes']['rank_16']:>15s} {hv_lora['lora_sizes']['rank_16']:>15s}")
print(f"{'Video editing':20s} {'VACE (native)':>15s} {'None':>15s}")

if wan_results:
    avg_wan = sum(r['time'] for r in wan_results) / len(wan_results)
    print(f"{'Avg gen time':20s} {avg_wan:>14.1f}s", end='')
if hv_results and 'time' in hv_results[0]:
    avg_hv = sum(r['time'] for r in hv_results if 'time' in r) / len([r for r in hv_results if 'time' in r])
    print(f" {avg_hv:>14.1f}s")
else:
    print(f" {'FAILED':>15s}")

In [ ]:
# Download results
!zip -r results.zip results/
from google.colab import files
files.download('results.zip')